<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

Lecture: SEU - FPGA HLS Design

Speaker: Kai Shao

Date: 2026/06/09

# 2 · Scheduling: making *How* a first-class citizen

In the lecture, the turning point from generic to domain-specific compilers was this:

> In a general compiler, **"How"** (tile? unroll? pipeline?) is decided *secretly inside
> the passes*. In a domain-specific compiler, **"How" is lifted out into an explicit
> object** you can describe, search, reuse, and compose.

Halide/TVM did this for CPUs/GPUs by separating **algorithm** from **schedule**. Allo
does it for hardware. This notebook is about the **schedule** half.

We will cover:

1. Why a *separate, composable* schedule matters (the "destructive pragma" problem).
2. Loop handles and how to grab them.
3. Loop transforms: `split`, `reorder`, `tile`, `flatten`, `unroll`, `pipeline`.
4. Memory/hardware transforms: `partition`, `buffer_at`, `reuse_at`.
5. **Composition** across kernels.
6. A **schedule sweep**: one algorithm → many hardware designs.

## 2.1 The destructive-pragma problem

In Vivado/Vitis HLS you optimize by editing the C source:

```c
void gemm(float A[16][16], float B[16][16], float C[16][16]) {
  for (int i = 0; i < 16; i++)
    for (int j = 0; j < 16; j++) {
      #pragma HLS pipeline II=1          // <-- schedule baked INTO the algorithm
      for (int k = 0; k < 16; k++)
        C[i][j] += A[i][k] * B[k][j];
    }
}
```

The pragma is **destructive**: it is welded to the algorithm. Want to try a *tiled* version
too? You copy-paste and edit the loops. Want to reuse this kernel's optimizations when it is
instantiated inside a larger design? You can't really — you rewrite C.

Allo keeps the algorithm pristine and expresses the schedule as a **separate program**:

```python
s = gemm.schedule()        # a fresh, independent schedule object
s.pipeline(s.loop("k"), ii=1)
```

The same `gemm` can feed *many* schedules, and schedules **compose** (§2.5). This is the
concrete payoff of "algorithm written once, How is composable and non-destructive."

In [ ]:
import numpy as np
import allo.exp as allo
from allo.exp.lang.kernel import kernel
from allo.exp.lang.core import i32, f32
from allo.exp.schedule import Schedule
from allo.exp.backend.vitis.core import is_vitis_available
print("Vitis available:", is_vitis_available())

# In VSCode/Jupyter, route Allo's logs to plain text instead of a live spinner
# widget. The spinner can render as an empty Output() in VSCode, making csim /
# synthesis look like it never starts (it is actually running underneath).
import allo.exp.logging as _allo_log
from rich.console import Console as _Console
_allo_log.console = _Console(stderr=True, force_interactive=False)

## 2.2 Grabbing loop handles

To transform a loop you first need a **handle** to it. Name the loops you intend to
schedule (`name="i"`), then query them:

* `s.loop("i")` — the single loop named `i`.
* `s.loops("i", "j", "k")` — several handles at once.
* `s.affine(s.loops(...))` — *raise* an `scf.for` band to `affine.for` so the
  polyhedral transforms (reorder/tile/flatten) become available, and return the handles.
* `s.buffer("A")` — a handle to a memory (memref) named `A`.

Transforms are recorded **lazily** and materialized on `apply()` (which `export()` calls
for you). `str(s.payload)` after `apply()` shows the transformed IR.

In [ ]:
M, N, K = 32, 64, 128

def fresh_gemm():
    @kernel
    def gemm(A: f32[M, K], B: f32[K, N], C: f32[M, N]):
        for i in allo.range(M, name="i"):
            for j in allo.range(N, name="j"):
                for k in allo.range(K, name="k"):
                    C[i, j] += A[i, k] * B[k, j]
    return gemm

s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
for loop in [i, j, k]:
    print("Loop:", loop)

## 2.3 Loop transforms

Every transform below leaves the **result unchanged** — only the *generated hardware*
changes. We verify correctness on the CPU after each one.

### Vanilla -- no transform

In [ ]:
s_gemm = fresh_gemm().schedule()
print(s_gemm.export("vitis").hls_code)

### `split` — break a loop into outer/inner

In [ ]:
s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
jo, ji = s.split(j, factor=8)        # 64 -> (8 outer) x (8 inner)
print("After split j:", s.export("vitis").hls_code)

### `reorder` — change loop nesting order

Loop order drives the memory access pattern (and thus reuse). For GEMM, moving `k` out
turns the inner `j` loop into a streaming row update.

In [ ]:
def run_and_check(s):
    mod = s.export("cpu")
    A = np.random.rand(M, K).astype(np.float32)
    B = np.random.rand(K, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    mod(A, B, C)
    ok = np.allclose(C, A @ B, rtol=1e-4)
    return ok

s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
s.reorder((i, k, j))                  # i, j, k  ->  i, k, j
print("reordered GEMM correct:", run_and_check(s))
print("After reorder i, k, j:", s.export("vitis").hls_code)

### `tile` — split two axes and interleave (locality)

`tile` is `split` + `reorder` fused: it blocks the iteration space into tiles. On FPGA
this controls the on-chip working-set size.

In [ ]:
s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
s.tile((i, j), factors=[8, 8])        # (i, j) -> (io, jo, ii, ji) with factors (8, 8)
print("tiled GEMM correct:", run_and_check(s))
print("After tile i, j with factors 8, 8:", s.export("vitis").hls_code)

### `flatten` — collapse a perfect nest into one loop

Useful before pipelining: a single flattened loop pipelines with no nested-loop flush
overhead.

In [ ]:
s = fresh_gemm().schedule()
i, j = s.loops("i", "j")
s.flatten((i, j))
print("flattened GEMM correct:", run_and_check(s))
print("After flatten i, j:", s.export("vitis").hls_code)

### `unroll` and `pipeline` — the two core hardware knobs

* **`pipeline(loop, ii=II)`** — overlap successive iterations so a new one starts every
  `II` cycles (Initiation Interval). `II=1` is the throughput ideal.
* **`unroll(loop, factor=f)`** — replicate the loop body `f` times → more parallel
  hardware (more DSPs/LUTs) per iteration.

These are recorded as attributes the HLS backend turns into pragmas — so the algorithm
stays clean.

In [ ]:
s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
s.reorder((i, k, j))
s.pipeline(j, ii=1)
print(s.export("vitis").hls_code)

s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
s.unroll(k, factor=4) # tag_only = True
print(s.export("vitis").hls_code)

s = fresh_gemm().schedule()
i, j, k = s.loops("i", "j", "k")
s.unroll(k, factor=4, tag_only=False)
print(s.export("vitis").hls_code)

## 2.4 Memory & hardware transforms

These have *no CPU-software analogue* — they are exactly the "extra axis of How" the
lecture said HLS adds (Slide 27): banking, on-chip buffering, reuse.

### `partition` — BRAM banking

An array stored in BRAM has a limited number of read/write ports. If a pipeline needs
several elements per cycle, the array must be **partitioned** into banks — otherwise the
pipeline II stalls on a memory-port conflict. `partition` records this as a hardware
attribute.

* `kind`: `Schedule.Complete` (every element its own register), `Schedule.Block`,
  `Schedule.Cyclic`.
* `dim`, `factor`: which dimension and how many banks.

In [ ]:
s = fresh_gemm().schedule()
s.partition(s.buffer("A"), dim=2, factor=8, kind=Schedule.Cyclic)
print(s.export("vitis").hls_code)

### `buffer_at` — stage data in on-chip memory

`buffer_at(buffer, axis)` makes a **private on-chip copy** of a buffer at a chosen loop
level, wrapping it with an automatic *copy-in → compute → copy-out*. On FPGA this is how
you keep a working set in fast on-chip RAM/registers instead of repeatedly reaching out to
off-chip memory.

The classic GEMM use: buffer the `C` accumulator at the output-row loop `i`. Combined with
`reorder` (accumulate across `k`, sweep `j` inside) and `pipeline`, the partial sums for a
whole output row stay on-chip and are written back only once per row.

In [ ]:
BM = BN = BK = 8

@kernel
def gemm_acc(A: f32[BM, BK], B: f32[BK, BN], C: f32[BM, BN]):
    for i in allo.range(BM, name="i"):
        for j in allo.range(BN, name="j"):
            for k in allo.range(BK, name="k"):
                C[i, j] += A[i, k] * B[k, j]

s = gemm_acc.schedule()
s.affine(s.loops("i", "j", "k"))
s.reorder((s.loop("k"), s.loop("j")))      # accumulate across k, sweep j inside
s.buffer_at(s.buffer("C"), s.loop("i"))    # private on-chip C row per output row
s.pipeline(s.loop("j"))
mod = s.export("cpu")

A = np.random.rand(BM, BK).astype(np.float32)
B = np.random.rand(BK, BN).astype(np.float32)
C = np.zeros((BM, BN), dtype=np.float32)
mod(A, B, C)
print("buffer_at GEMM correct:", np.allclose(C, A @ B, rtol=1e-4))

### `compute_at` — fuse a producer into its consumer

When one loop nest *produces* a buffer that a later nest *consumes*, `compute_at(producer,
consumer)` moves the producer's computation **inside** the consumer's loop. This is loop
**fusion**: the intermediate no longer has to be fully materialized, the two passes
traverse the data once, and — crucially for HLS — the stages can be **streamed / pipelined
together** instead of running one-after-another through a large intermediate buffer.

Below, a producer writes `B = A + 1` and a consumer writes `C = B * 2`. `compute_at` fuses
the producer's `bi` band into the consumer's `ci` band, so the two output loop nests
collapse into one (watch the `affine.for` count drop).

In [ ]:
FH, FW = 8, 8

@kernel
def two_stage(A: i32[FH, FW], C: i32[FH, FW]):
    B: i32[FH, FW] = 0
    for bi in allo.range(FH, name="bi"):          # producer band: B = A + 1
        for bj in allo.range(FW, name="bj"):
            B[bi, bj] = A[bi, bj] + 1
    for ci in allo.range(FH, name="ci"):          # consumer band: C = B * 2
        for cj in allo.range(FW, name="cj"):
            C[ci, cj] = B[ci, cj] * 2

n_before = str(two_stage.schedule().payload).count("affine.for")

s = two_stage.schedule()
s.affine(s.loops("bi", "bj", "ci", "cj"))
s.compute_at(s.loop("bi"), s.loop("ci"))      # fuse producer into the consumer loop
s.apply()
n_after = str(s.payload).count("affine.for")
mod = s.export("cpu")

A = np.random.randint(0, 10, (FH, FW)).astype(np.int32)
C = np.zeros((FH, FW), dtype=np.int32)
mod(A, C)
print("compute_at fused result correct:", np.array_equal(C, (A + 1) * 2))
print(f"affine.for nests: {n_before} -> {n_after} (the two bands fused into one)")

### `reuse_at` — sliding-window reuse (stencils)

For a stencil that reads overlapping neighbourhoods, `reuse_at` synthesizes a
**line/window buffer** so each input element is read once and shifted through, instead of
re-read for every output. This is the classic HLS reuse-buffer, generated for you.

In [ ]:
H, W = 16, 16

@kernel
def blur(A: i32[H, W], B: i32[H, 14]):
    for y in allo.range(H, name="y"):
        for x in allo.range(14, name="x"):
            B[y, x] = A[y, x] + A[y, x + 1] + A[y, x + 2]

s = blur.schedule()
s.reuse_at(s.buffer("A"), s.loop("x"))     # sliding window over x
mod = s.export("cpu")

A = np.random.randint(0, 10, (H, W)).astype(np.int32)
B = np.zeros((H, 14), dtype=np.int32)
mod(A, B)
ref = A[:, 0:14] + A[:, 1:15] + A[:, 2:16]
print("reuse_at stencil correct:", np.array_equal(B, ref))
print(s.export("vitis").hls_code)

## 2.5 Composition — the non-destructive payoff

Because schedules are independent objects, you can schedule each kernel **separately**
and then **compose** them into a larger design. Nothing is rewritten; the optimizations
travel with the kernel. This is what destructive pragmas cannot do.

In [ ]:
SZ = 16

@kernel
def gemm(A: i32[SZ, SZ], B: i32[SZ, SZ], C: i32[SZ, SZ]):
    for i in allo.range(SZ, name="i"):
        for j in allo.range(SZ, name="j"):
            for k in allo.range(SZ, name="k"):
                C[i, j] += A[i, k] * B[k, j]

@kernel
def addone(C: i32[SZ, SZ], D: i32[SZ, SZ]):
    for i in allo.range(SZ, name="i"):
        for j in allo.range(SZ, name="j"):
            D[i, j] = C[i, j] + 1

@kernel
def top(A: i32[SZ, SZ], B: i32[SZ, SZ], C: i32[SZ, SZ], D: i32[SZ, SZ]):
    gemm(A, B, C)
    addone(C, D)

# Schedule each stage on its own ...
gs = gemm.schedule(); gs.pipeline(gs.loop("j"), ii=1)
as_ = addone.schedule(); as_.pipeline(as_.loop("j"), ii=1)

# ... then compose both into the top-level pipeline.
ts = top.schedule()
ts.compose(gs, as_)
mod = ts.export("cpu")

A = np.random.randint(0, 5, (SZ, SZ)).astype(np.int32)
B = np.random.randint(0, 5, (SZ, SZ)).astype(np.int32)
C = np.zeros((SZ, SZ), dtype=np.int32)
D = np.zeros((SZ, SZ), dtype=np.int32)
mod(A, B, C, D)
print("composed pipeline correct:", np.array_equal(D, A @ B + 1))
print(ts.export("vitis").hls_code)

## 2.6 One algorithm, many hardware designs — a schedule sweep

This is the demo the lecture asked for (Slide 29): *same payload + different schedule →
different hardware.* We keep `gemm` fixed and sweep over a handful of schedules.

### (a) Fast structural sweep (no toolchain)

Even without synthesis we can *see* the structure change: the emitted HLS differs in
loop count, pipeline pragmas, and unroll pragmas.

In [ ]:
GM = GN = GK = 16

def make_gemm():
    @kernel
    def gemm(A: f32[GM, GK], B: f32[GK, GN], C: f32[GM, GN]):
        for i in allo.range(GM, name="i"):
            for j in allo.range(GN, name="j"):
                for k in allo.range(GK, name="k"):
                    C[i, j] += A[i, k] * B[k, j]
    return gemm

def sched_naive(s):
    return s                                   # baseline, no transforms

def sched_pipe(s):
    s.reorder((s.loop("i"), s.loop("k"), s.loop("j")))
    s.pipeline(s.loop("j"), ii=1)
    return s

def sched_tile_pipe(s):
    i, j, k = s.loops("i", "j", "k")
    s.tile((i, j), factors=[4, 4])
    s.pipeline(s.loop("k"), ii=1)
    return s

def sched_unroll(s):
    i, j, k = s.loops("i", "j", "k")
    s.reorder((i, k, j))
    s.pipeline(j, ii=1)
    s.unroll(k, factor=4)
    return s

VARIANTS = {
    "naive":      sched_naive,
    "pipeline":   sched_pipe,
    "tile+pipe":  sched_tile_pipe,
    "unroll+pipe": sched_unroll,
}

for name, fn in VARIANTS.items():
    code_str = fn(make_gemm().schedule()).export("vitis").hls_code
    print(code_str)

### (b) Real QoR sweep (requires Vitis HLS)

The structural view is suggestive, but the real point is *quality of results* (QoR):
latency, II, and resource usage. The cell below runs **actual Vitis HLS synthesis** for
each variant and tabulates the numbers parsed from the synthesis report.

> ⚠️ Synthesis is slow — expect a couple of minutes **per** variant. It is gated on a
> detected Vitis toolchain; skip it if you only have the software simulator.

This is exactly where the lecture's warning bites (Slide 20): schedule primitives
**interact**, and an *isolated* primitive can be a **pessimization**. Look at the latency
column once synthesis finishes: naively adding `pipeline` to the reordered loop is
*dramatically worse* than the plain `naive` loop (the inner accumulation cannot reach a
low II, so iterations serialize), while `tile+pipe` — pipelining *inside* a blocked
iteration space — is the clear winner. `unroll+pipe` buys parallel DSPs/LUTs but, on its
own, does not fix the II problem. The lesson: **the best design is a *combination*, not
any single knob** — which is precisely why design-space exploration and cost models exist.

In [ ]:
import tempfile, xml.etree.ElementTree as ET
from pathlib import Path

def synth_qor(name, sched_fn, part="xcvu9p-flga2104-2-i"):
    s = sched_fn(make_gemm().schedule())
    with tempfile.TemporaryDirectory() as proj:
        report = s.export("vitis", part=part, project_path=proj).synth()
        root = ET.parse(report.xml_path).getroot()
        lat = root.find("./PerformanceEstimates/SummaryOfOverallLatency")
        res = root.find("./AreaEstimates/Resources")
        def g(node, tag):
            el = node.find(tag); return el.text if el is not None else "-"
        return {
            "schedule": name,
            "latency": g(lat, "Average-caseLatency"),
            "DSP": g(res, "DSP"),
            "LUT": g(res, "LUT"),
            "FF":  g(res, "FF"),
            "BRAM": g(res, "BRAM_18K"),
        }

if is_vitis_available():
    rows = [synth_qor(n, fn) for n, fn in VARIANTS.items()]
    cols = ["schedule", "latency", "DSP", "LUT", "FF", "BRAM"]
    print("".join(f"{c:>11}" for c in cols))
    for r in rows:
        print("".join(f"{str(r[c]):>11}" for c in cols))
else:
    print("Vitis HLS not available -- skipping the real QoR sweep.")

## Recap

* The schedule is a **separate, composable object** — the algorithm is never edited.
  This is the cure for destructive pragmas.
* Loop transforms (`split`/`reorder`/`tile`/`flatten`/`unroll`/`pipeline`) and
  memory/hardware transforms (`partition`/`buffer_at`/`reuse_at`) are the vocabulary of
  *How* — including the FPGA-only axis (banking, reuse buffers) a CPU compiler lacks.
* `compose` lets per-kernel optimizations survive into a larger design.
* **Same payload + different schedule → different hardware**, and the knobs **interact** —
  which is exactly why design-space exploration (and cost models) matter.

**Next:** [`03_simulation.ipynb`](03_simulation.ipynb) — verifying correctness fast,
before committing to synthesis.